# Steam Trajectory — Scrape & Database Load

Reads the candidate list frozen by `00_cohort_selection.ipynb`, scrapes SteamCharts for each candidate's monthly history, drops any that fail, freezes the resulting **final cohort** to `cohort_appids.csv` (so downstream notebooks never depend on live network conditions again), and loads everything into `steam_project.db`.

This notebook is standalone: it never re-derives the cohort from the Kaggle metadata, and never needs to reload the full ~126K-game JSON dataset.

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("/Users/pmacias/Dropbox/steamproject")
print(os.getcwd())

/Users/pmacias/Dropbox/steamproject


In [2]:
import pandas as pd

from steam_trajectory.ingest.steamcharts_scraper import SteamChartsScraper
from steam_trajectory.ingest.kaggle_loader import KaggleLoader
from steam_trajectory.db.connection import get_connection
from steam_trajectory.db.writer import DatabaseWriter

In [3]:
candidates = pd.read_csv("candidate_appids.csv")
print(f"Loaded {len(candidates)} candidates")
candidates.head()

Loaded 260 candidates


,AppID,Name,Release date,Developers,Publishers,Price,total_reviews,review_score_percent,Genres
0,1174390,ANNIE:Last Hope,"Apr 6, 2020",Pixel Rice,Pixel Rice,5.99,618,92.071197,"Action, Adventure, Indie, RPG"
1,1893620,Circadian Dice,"Jul 11, 2022",Shuffle Up Games,Shuffle Up Games,4.79,578,94.463668,Strategy
2,1057090,Ori and the Will of the Wisps,"Mar 10, 2020",Moon Studios GmbH,Xbox Game Studios,7.49,136690,96.560831,Action
3,1035560,Struggling,"Aug 27, 2020",Chasing Rats Games,Frontier Foundry,3.74,681,86.784141,"Action, Adventure, Casual, Indie"
4,1946550,Toilet Chronicles,"Jul 14, 2022",Madi Abdykarimov,Bomi Games,0.99,1389,90.064795,"Action, Adventure, Indie, Simulation"


## Quick scraper sanity check
Optional — a fast check that the scraper still works before committing to the full batch. Safe to skip on reruns once you trust it.

In [6]:
scraper = SteamChartsScraper()
test_appids = candidates["AppID"].head(3).tolist()
print("Testing:", test_appids)

test_histories, test_failures = scraper.get_monthly_history_batch(test_appids)
print(f"\n{len(test_histories)} monthly records, {len(test_failures)} failures")
print(test_failures)


Testing: [1174390, 1893620, 1057090]

76 monthly records, 2 failures
[{'appid': 1174390, 'error': '500 Server Error: Internal Server Error for url: https://steamcharts.com/app/1174390'}, {'appid': 1893620, 'error': '500 Server Error: Internal Server Error for url: https://steamcharts.com/app/1893620'}]


## Full scrape + freeze final cohort
Scrapes every candidate, drops any that fail (no SteamCharts page, timeout, etc. — see notes in `steamcharts_scraper.py`), and **freezes the resulting cohort to `cohort_appids.csv`**. This is the fix for the reproducibility gap we hit earlier: since scraping failures depend partly on live network/server conditions, rerunning this cell can legitimately produce a different-sized survivor set each time. Freezing the result means every notebook from here on reads a fixed list, rather than silently re-deriving a possibly-different cohort on every rerun.

In [7]:
all_appids = candidates["AppID"].tolist()
all_histories, failures = scraper.get_monthly_history_batch(all_appids)

print(f"\nDone. {len(all_histories)} monthly records, {len(failures)} candidates failed.")
print(failures)

Processed 20/260 games (4 failures so far)...
Processed 40/260 games (9 failures so far)...
Processed 60/260 games (10 failures so far)...
Processed 80/260 games (14 failures so far)...
Processed 100/260 games (20 failures so far)...
Processed 120/260 games (27 failures so far)...
Processed 140/260 games (30 failures so far)...
Processed 160/260 games (35 failures so far)...
Processed 180/260 games (45 failures so far)...
Processed 200/260 games (56 failures so far)...
Processed 220/260 games (61 failures so far)...
Processed 240/260 games (65 failures so far)...
Processed 260/260 games (71 failures so far)...

Done. 12393 monthly records, 71 candidates failed.
[{'appid': 1174390, 'error': '500 Server Error: Internal Server Error for url: https://steamcharts.com/app/1174390'}, {'appid': 1893620, 'error': '500 Server Error: Internal Server Error for url: https://steamcharts.com/app/1893620'}, {'appid': 1946550, 'error': '500 Server Error: Internal Server Error for url: https://steamchar

In [8]:
failed_appids = {f["appid"] for f in failures}
final_cohort = candidates[~candidates["AppID"].isin(failed_appids)].reset_index(drop=True)

final_appids = set(final_cohort["AppID"])
final_histories = [record for record in all_histories if record["appid"] in final_appids]

print(f"Final cohort size: {len(final_cohort)}")
print(f"Monthly records for final cohort: {len(final_histories)}")

final_cohort.to_csv("cohort_appids.csv", index=False)
print("Saved final cohort to cohort_appids.csv")

Final cohort size: 189
Monthly records for final cohort: 12393
Saved final cohort to cohort_appids.csv


## Load into the database
`KaggleLoader.iter_games` / `iter_genres` are called as staticmethods here — no `KaggleLoader` instance needed, so this never reloads the full JSON dataset.

In [9]:
conn = get_connection("steam_project.db")
writer = DatabaseWriter(conn)

# Clear all three tables tied to cohort membership before reloading —
# order matters due to foreign key constraints: monthly_metrics and
# game_genres both reference games.appid, so they must be cleared
# before games itself. (genres, the lookup table, doesn't need
# clearing — genre names are reused across cohort versions, not
# tied to any specific cohort.)
conn.execute("DELETE FROM monthly_metrics")
conn.execute("DELETE FROM game_genres")
conn.execute("DELETE FROM games")
conn.commit()

# 1. Games
for game in KaggleLoader.iter_games(final_cohort):
    writer.insert_game(**game)

# 2. Genre links
for appid, genre_name in KaggleLoader.iter_genres(final_cohort):
    writer.link_game_genre(appid, genre_name)

# 3. Monthly metrics from the scrape
for record in final_histories:
    writer.insert_monthly_metric(**record)

writer.commit()
print("Database load complete.")

Database load complete.


## Verify the load

In [10]:
print("games:", pd.read_sql("SELECT COUNT(*) as n FROM games", conn)["n"][0])
print("game_genres:", pd.read_sql("SELECT COUNT(*) as n FROM game_genres", conn)["n"][0])
print("monthly_metrics:", pd.read_sql("SELECT COUNT(*) as n FROM monthly_metrics", conn)["n"][0])

pd.read_sql("""
    SELECT g.name, g.release_date, gen.name AS genre, COUNT(m.month) AS months_tracked
    FROM games g
    JOIN game_genres gg ON g.appid = gg.appid
    JOIN genres gen ON gg.genre_id = gen.genre_id
    JOIN monthly_metrics m ON g.appid = m.appid
    GROUP BY g.appid, gen.name
    LIMIT 10
""", conn)

games: 189
game_genres: 531
monthly_metrics: 12393


,name,release_date,genre,months_tracked
0,Grand Theft Auto IV: The Complete Edition,"Mar 24, 2020",Action,168
1,Grand Theft Auto IV: The Complete Edition,"Mar 24, 2020",Adventure,168
2,Serious Sam Classic: The Second Encounter,"Aug 30, 2019",Action,168
3,Serious Sam Classic: The Second Encounter,"Aug 30, 2019",Indie,168
4,Imagine Earth,"May 25, 2021",Casual,147
5,Imagine Earth,"May 25, 2021",Indie,147
6,Imagine Earth,"May 25, 2021",Simulation,147
7,Imagine Earth,"May 25, 2021",Strategy,147
8,On The Road - The Truck Simulator,"Nov 14, 2019",Casual,107
9,On The Road - The Truck Simulator,"Nov 14, 2019",Simulation,107


In [11]:
frozen_appids = set(pd.read_csv("cohort_appids.csv")["AppID"])
db_appids = set(pd.read_sql("SELECT appid FROM games", conn)["appid"])
print("Match:",  frozen_appids == db_appids)


Match: True
